# Experimento 10 — Benchmark de classificadores para detectar mixers Ethereum

**Pergunta:** algum classificador supera a Random Forest original sob o *mesmo* conjunto de features e protocolo de avaliação?

**Status científico:** experimento exploratório. As amostras positivas de treino são 23 janelas de **um único pool Tornado Cash (10 ETH)**. Validação por linha pode produzir resultados otimistas. **Não alegar generalização a novos contratos**; não selecionar modelos nem limiares usando o teste histórico reservado de 2026.

Modelos: Random Forest original, Extra Trees, HistGradientBoosting e Regressão Logística. Comparação interna: validação cruzada estratificada repetida, com oversampling somente dentro do fold de treino, mesmas divisões para os quatro modelos. Métricas: AP, F1, precisão, recall, número de falsos positivos; limiar 0,5 apenas para diagnóstico.

**Entrada:** `MyDrive/mixers-ethereum/data/processed/features_treino.parquet` (ou ajuste a variável BASE abaixo). **Saídas:** em `results/10_benchmark_exploratorio/`. Nenhum arquivo original é alterado.

In [11]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

for pasta in sorted(DRIVE.iterdir()):
    if "mixers-ethereum" in pasta.name.lower():
        print(f"\n📁 {pasta.name}")

        arquivo = pasta / "data/processed/features_treino.parquet"

        if arquivo.exists():
            print("✅ DATASET ENCONTRADO!")
            print("Caminho:", arquivo)
            print("Tamanho:", arquivo.stat().st_size, "bytes")
        else:
            print("❌ Dataset não encontrado nesse caminho")

            if pasta.is_dir():
                print("Conteúdo:", [p.name for p in pasta.iterdir()][:15])


📁 mixers-ethereum
❌ Dataset não encontrado nesse caminho
Conteúdo: ['results']

📁 mixers-ethereum (1)
✅ DATASET ENCONTRADO!
Caminho: /content/drive/MyDrive/mixers-ethereum (1)/data/processed/features_treino.parquet
Tamanho: 13025 bytes

📁 mixers-ethereum (2)
✅ DATASET ENCONTRADO!
Caminho: /content/drive/MyDrive/mixers-ethereum (2)/data/processed/features_treino.parquet
Tamanho: 13025 bytes


In [12]:
import pandas as pd, numpy as np, sklearn, json
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.utils import resample
FEATURES=['value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
df=pd.read_parquet(DATA)
assert set(FEATURES+['label']).issubset(df.columns), ('Colunas disponíveis:',df.columns.tolist())
df=df.copy()
for f in FEATURES:df[f]=pd.to_numeric(df[f],errors='coerce')
df['label']=df['label'].astype(int)
assert set(df.label.unique())=={0,1}, 'Esperadas exatamente classes 0 e 1'
assert df[FEATURES].notna().all().all(), 'Dados com valores ausentes; examine antes de modelar'
print('Amostras:',len(df),'Classes:',df.label.value_counts().to_dict())
print('Colunas:',df.columns.tolist())
print('Versão sklearn:',sklearn.__version__)
print('ATENÇÃO: positivos de um único contrato, validação interna por linha não estima generalização entre contratos.')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/mixers-ethereum/data/processed/features_treino.parquet'

In [ ]:
# Auditoria de grupos: bloqueia falsa alegação de validação por contrato
for col in ['recipient','address','contract','contract_address']:
    if col in df.columns:
        print('Grupos por classe em',col,df.groupby('label')[col].nunique().to_dict())
        break
else:
    print('Sem ID de contrato no dataset. Documentação da coleta: classe positiva = mesmo contrato Tornado 10 ETH em dias distintos.')
# Duplicação da matriz de features pode aumentar otimismo
print('Linhas duplicadas nas quatro features e rótulo:',df[FEATURES+['label']].duplicated().sum())
print('Positivos distintos no espaço das quatro features:',df.loc[df.label==1,FEATURES].drop_duplicates().shape[0])

In [ ]:
# Configuração PRÉ-DEFINIDA (não usar 2020/2026 para escolher hiperparâmetros)
SEEDS=[11,23,42,77,101]
K=5

def criar_modelos(seed):
    return {
      'Random Forest (artigo)': RandomForestClassifier(n_estimators=100,max_depth=5,class_weight='balanced',random_state=seed,n_jobs=-1),
      'Extra Trees': ExtraTreesClassifier(n_estimators=100,max_depth=5,class_weight='balanced',random_state=seed,n_jobs=-1),
      'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,l2_regularization=1.0,learning_rate=0.05,min_samples_leaf=5,random_state=seed),
      'Regressão Logística': make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=2000,random_state=seed))
    }

def oversampling_treino(X,y,seed):
    base=X.copy().reset_index(drop=True)
    base['_y']=np.asarray(y)
    maj=base.loc[base._y==0]
    minor=base.loc[base._y==1]
    # procedimento do experimento original; apenas fold de treino
    if len(minor)<len(maj):
        minor=resample(minor,replace=True,n_samples=len(maj),random_state=seed)
    b=pd.concat([maj,minor],ignore_index=True)
    return b[FEATURES],b['_y']

X=df[FEATURES];y=df.label
assert y.value_counts().min()>=K
print('Plano:',len(SEEDS),'sementes x',K,'folds x 4 modelos')

In [ ]:
# Executa experimentos idênticos entre algoritmos, sem olhar o teste externo
records=[]
for seed in SEEDS:
    cv=StratifiedKFold(n_splits=K,shuffle=True,random_state=seed)
    for fold,(itr,iva) in enumerate(cv.split(X,y),1):
        Xt,yt=oversampling_treino(X.iloc[itr],y.iloc[itr],seed+fold)
        Xv,yv=X.iloc[iva],y.iloc[iva]
        for nome,model in criar_modelos(seed).items():
            model.fit(Xt,yt)
            p=model.predict_proba(Xv)[:,1]
            pred=(p>0.5).astype(int)
            tn,fp,fn,tp=confusion_matrix(yv,pred,labels=[0,1]).ravel()
            records.append(dict(modelo=nome,semente=seed,fold=fold,n_validacao=len(yv),positivos_validacao=int(yv.sum()),
                                ap=average_precision_score(yv,p),precision=precision_score(yv,pred,zero_division=0),
                                recall=recall_score(yv,pred,zero_division=0),f1=f1_score(yv,pred,zero_division=0),
                                tp=int(tp),fp=int(fp),fn=int(fn),tn=int(tn)))
metricas=pd.DataFrame(records)
metricas.to_csv(OUT/'cv_folds.csv',index=False)
print('Rodadas completadas:',len(metricas));display(metricas.head())

In [ ]:
# Agregação por semente; depois média e desvio entre sementes
por_semente=metricas.groupby(['modelo','semente'],as_index=False).agg(
  ap=('ap','mean'), precision=('precision','mean'), recall=('recall','mean'), f1=('f1','mean'),
  fp_total=('fp','sum'),fn_total=('fn','sum'),tp_total=('tp','sum'))
por_semente.to_csv(OUT/'cv_por_semente.csv',index=False)
resumo=por_semente.groupby('modelo').agg(
    f1_medio=('f1','mean'),f1_desvio=('f1','std'),ap_medio=('ap','mean'),ap_desvio=('ap','std'),
    precision_media=('precision','mean'),recall_medio=('recall','mean'),
    fp_total_medio=('fp_total','mean'),fn_total_medio=('fn_total','mean')).sort_values('ap_medio',ascending=False)
resumo.to_csv(OUT/'benchmark_resumo.csv',encoding='utf-8-sig')
display(resumo.round(3))
print('ATENÇÃO: comparações de validação interna; nenhum modelo vencedor para novos contratos.')

In [ ]:
# Gráfico; barras de erro representam variação entre as 5 sementes
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,4.8))
r=resumo.sort_values('ap_medio')
ax.barh(r.index,r.ap_medio,xerr=r.ap_desvio,alpha=.85,capsize=4)
ax.set_xlim(0,1.05); ax.set_xlabel('Average Precision médio na validação interna')
ax.set_title('Benchmark exploratório — dados positivos do mesmo pool')
fig.tight_layout();fig.savefig(OUT/'benchmark_ap.png',dpi=170)
plt.show()

## Interpretação e restrições

1. **Não afirmar que o modelo líder generaliza:** o mesmo contrato Tornado 10 ETH aparece em diferentes linhas/folds. É necessário coletar exemplos positivos de múltiplos contratos independentes para fazer validação por grupos.
2. **Não otimizar com 2026:** 15/01/2026 já foi usado em análises e precisa ser reportado como *teste histórico explorado*, não como novo teste intocado.
3. **Não calcular precisão global com rótulos desconhecidos:** ausência de endereço na lista original de Tornado não prova que seja negativo.
4. **Não afirmar que vários modelos são independentes:** todos aprendem as mesmas quatro features; votar não necessariamente reduz erros correlacionados.
5. **Próximo experimento rigoroso:** congelar dados novos de desenvolvimento, rótulos por contrato, divisões por contrato, métricas e regras de limiar; depois comparar classificação individual e votação **somente na validação**. Reservar novo período, nunca visto, para avaliação final.

**Arquivos:** `cv_folds.csv`, `cv_por_semente.csv`, `benchmark_resumo.csv`, `benchmark_ap.png`. Não substituir os relatórios anteriores.